In [27]:
import pandas as pd

In [26]:
1+1

2

In [28]:
import psycopg2

# Подключение к базе данных
conn = psycopg2.connect(
    dbname="de",
    user="jovyan",
    password="jovyan",
    host="localhost",
    port="15432"
)

# Создание курсора для выполнения SQL запросов
cur = conn.cursor()

# Выполнение SELECT запроса
cur.execute(" select schema_name from information_schema.schemata")

# Получение результатов запроса
rows = cur.fetchall()

# Вывод результатов
for row in rows:
    print(row)

# Закрытие курсора и соединения
cur.close()
conn.close()

('pg_toast',)
('pg_catalog',)
('public',)
('information_schema',)
('mart',)
('stage',)


In [29]:
user_activity_log = pd.read_csv(f'user_activity_log.csv', index_col=0)

In [30]:
user_activity_log.head(5)

,uniq_id,date_time,action_id,customer_id,quantity
id,,,,,
1,b369e8ef7f08e908e47c396aabb0c800,2024-12-28T00:00:00,1,611,6
2,a1bfd8b7d2a00f0690dc9ce399424ab0,2024-12-28T00:00:00,3,421,18
3,e3ee959bc7fab74bec436fcfcad3f29f,2024-12-28T00:00:00,3,980,21
4,9c37b38cb9c6764932bdc735a8b11f6e,2024-12-28T00:00:00,3,110,9
5,110cf464cbe8213850ae7345cf7d1d0f,2024-12-28T00:00:00,1,550,16


In [31]:
user_activity_log = user_activity_log.drop(['uniq_id'], axis=1)

In [32]:
user_activity_log.head(5)

,date_time,action_id,customer_id,quantity
id,,,,
1,2024-12-28T00:00:00,1,611,6
2,2024-12-28T00:00:00,3,421,18
3,2024-12-28T00:00:00,3,980,21
4,2024-12-28T00:00:00,3,110,9
5,2024-12-28T00:00:00,1,550,16


In [33]:
user_activity_log

,date_time,action_id,customer_id,quantity
id,,,,
1,2024-12-28T00:00:00,1,611,6
2,2024-12-28T00:00:00,3,421,18
3,2024-12-28T00:00:00,3,980,21
4,2024-12-28T00:00:00,3,110,9
5,2024-12-28T00:00:00,1,550,16
...,...,...,...,...
220,2025-01-19T00:00:00,1,671,6
221,2025-01-19T00:00:00,1,961,24
222,2025-01-19T00:00:00,2,890,21


In [34]:
user_order_log = pd.read_csv(f'user_order_log.csv', index_col=0)

In [35]:
user_order_log = user_order_log.drop(columns='uniq_id')

In [36]:
user_order_log[user_order_log['customer_id']==11]

,date_time,city_id,city_name,customer_id,first_name,last_name,item_id,item_name,quantity,payment_amount
id,,,,,,,,,,
20218,2025-01-17T00:00:00,0,Москва,11,Александра,Великая,10,Томаты,1,50
20440,2025-01-17T00:00:00,0,Москва,11,Александра,Великая,10,Томаты,1,50
20662,2025-01-17T00:00:00,0,Москва,11,Александра,Великая,10,Томаты,1,50
20884,2025-01-17T00:00:00,0,Москва,11,Александра,Великая,10,Томаты,1,50


In [37]:
customer_research = pd.read_csv(f'customer_research.csv')

In [38]:
customer_research

,date_id,category_id,geo_id,sales_qty,sales_amt
0,2024-12-28T00:00:00,1001,1,5,5090
1,2024-12-28T00:00:00,1001,2,3,3054
2,2024-12-28T00:00:00,1001,3,7,7126
3,2024-12-28T00:00:00,1002,1,5,5090
4,2024-12-28T00:00:00,1002,2,5,5090
...,...,...,...,...,...
685,2025-01-19T00:00:00,1009,2,1,1018
686,2025-01-19T00:00:00,1009,3,3,3054
687,2025-01-19T00:00:00,1010,1,5,5090
688,2025-01-19T00:00:00,1010,2,6,6108


In [39]:
import psycopg2
def ex_sql(s):
    conn = psycopg2.connect("dbname='de' port='15432' user='jovyan' host='localhost' password='jovyan'")
    cur = conn.cursor()
    cur.execute(s)
    conn.commit()
    cur.close()
    conn.close()   
    return None
ex_sql("select 1")

In [51]:
s= '''
-- for test
DROP SCHEMA IF EXISTS stage CASCADE;
CREATE SCHEMA stage;

--clear
-- DROP TABLE IF EXISTS stage.user_activity_log;
-- DROP TABLE IF EXISTS stage.user_order_log;
-- DROP TABLE IF EXISTS stage.customer_research;


--user activity log
CREATE TABLE stage.user_activity_log(
   ID serial ,
 --uniq_id text,
   date_time          TIMESTAMP ,
   action_id             BIGINT ,
   customer_id             BIGINT ,
   quantity             BIGINT ,
   PRIMARY KEY (ID)
);

CREATE INDEX main1 ON stage.user_activity_log (customer_id);

--user order log
CREATE TABLE stage.user_order_log(
   ID serial ,
 --  uniq_id text,
   date_time          TIMESTAMP ,
   city_id integer,
   city_name varchar (100),
   customer_id             BIGINT ,
   first_name varchar (100),
   last_name varchar (100),
   item_id integer,
   item_name    varchar(100),
   quantity             BIGINT ,
   payment_amount numeric(14,2),

   PRIMARY KEY (ID)
);

CREATE INDEX main2 ON stage.user_order_log (customer_id);


--customer research
CREATE TABLE stage.customer_research(
   ID serial,
   date_id          TIMESTAMP ,
   category_id integer,
   geo_id   integer,
   sales_qty    integer,
   sales_amt numeric(14,2),


   PRIMARY KEY (ID)
);

CREATE INDEX main3 ON stage.customer_research (category_id);


select * from stage.user_activity_log;



'''

In [52]:
ex_sql(s)

In [42]:
insert_s = '''insert into stage.user_order_log (
 date_time, city_id, city_name, customer_id, first_name,
last_name, item_id, item_name, quantity,payment_amount

)
VALUES {s_val};'''

In [43]:
user_order_log.columns

Index(['date_time', 'city_id', 'city_name', 'customer_id', 'first_name',
       'last_name', 'item_id', 'item_name', 'quantity', 'payment_amount'],
      dtype='object')

In [44]:
user_order_log

,date_time,city_id,city_name,customer_id,first_name,last_name,item_id,item_name,quantity,payment_amount
id,,,,,,,,,,
1,2024-12-28T00:00:00,1,Ростов,3501,Олег,Попов,6,Яблоко,1,140
2,2024-12-28T00:00:00,0,Москва,7202,Константин,Сидоров,7,Средство для мытья посуды,3,600
3,2024-12-28T00:00:00,1,Ростов,4506,Сергей,Попов,6,Яблоко,2,280
4,2024-12-28T00:00:00,1,Ростов,6016,Елизавета,Великая,6,Яблоко,4,560
5,2024-12-28T00:00:00,0,Москва,9611,Надежда,Егорова,6,Яблоко,4,560
...,...,...,...,...,...,...,...,...,...,...
22992,2025-01-19T00:00:00,0,Москва,7510,Анна,Лебедева,5,Яблоко молодильное,2,800
22993,2025-01-19T00:00:00,2,Ярославль,8911,Снежанна,Орлова,10,Томаты,3,150
22994,2025-01-19T00:00:00,2,Ярославль,6312,Елизавета,Волкова,11,Хлеб ржаной,1,20


In [45]:
user_orders_log=user_order_log
i = 0
step = int(user_order_log.shape[0] / 100)
while i <= user_order_log.shape[0]:
    print(i, end='\r')
    
    s_val = str([tuple(x) for x in user_order_log.loc[i:i + step].to_numpy()])[1:-1]
    ex_sql(insert_s.replace('{s_val}',s_val))
    i += step+1

In [46]:
insert_s = '''insert into stage.user_activity_log (
 date_time, action_id, customer_id, quantity

)
VALUES {s_val};'''

In [47]:
i = 0
step = int(user_activity_log.shape[0] / 100)
while i <= user_activity_log.shape[0]:
    print(i, end='\r')
    s_val = str([tuple(x) for x in user_activity_log.loc[i:i + step].to_numpy()])[1:-1]
    print(s_val)
    ex_sql(insert_s.replace('{s_val}',s_val))
    i += step+1

('2024-12-28T00:00:00', 1, 611, 6), ('2024-12-28T00:00:00', 3, 421, 18)
('2024-12-28T00:00:00', 3, 980, 21), ('2024-12-28T00:00:00', 3, 110, 9), ('2024-12-28T00:00:00', 1, 550, 16)
('2024-12-28T00:00:00', 2, 711, 10), ('2024-12-28T00:00:00', 2, 661, 13), ('2024-12-29T00:00:00', 3, 440, 24)
('2024-12-29T00:00:00', 3, 820, 18), ('2024-12-29T00:00:00', 1, 661, 4), ('2024-12-29T00:00:00', 0, 431, 12)
('2024-12-29T00:00:00', 2, 530, 11), ('2024-12-29T00:00:00', 0, 380, 12), ('2024-12-29T00:00:00', 3, 340, 7)
('2024-12-29T00:00:00', 2, 541, 15), ('2024-12-30T00:00:00', 0, 770, 15), ('2024-12-31T00:00:00', 3, 600, 8)
('2024-12-31T00:00:00', 1, 751, 7), ('2024-12-31T00:00:00', 0, 640, 21), ('2024-12-31T00:00:00', 1, 491, 8)
('2024-12-31T00:00:00', 1, 100, 2), ('2024-12-31T00:00:00', 3, 750, 17), ('2024-12-31T00:00:00', 3, 291, 0)
('2024-12-31T00:00:00', 0, 560, 23), ('2024-12-31T00:00:00', 0, 241, 5), ('2025-01-01T00:00:00', 2, 880, 12)
('2025-01-01T00:00:00', 1, 501, 24), ('2025-01-01T00:00:0

In [48]:
user_activity_log

,date_time,action_id,customer_id,quantity
id,,,,
1,2024-12-28T00:00:00,1,611,6
2,2024-12-28T00:00:00,3,421,18
3,2024-12-28T00:00:00,3,980,21
4,2024-12-28T00:00:00,3,110,9
5,2024-12-28T00:00:00,1,550,16
...,...,...,...,...
220,2025-01-19T00:00:00,1,671,6
221,2025-01-19T00:00:00,1,961,24
222,2025-01-19T00:00:00,2,890,21


In [49]:
insert_s = '''insert into stage.customer_research (
date_id, category_id, geo_id, sales_qty, sales_amt

)
VALUES {s_val};'''

In [50]:

step = int(customer_research.shape[0] / 100)
while i <= customer_research.shape[0]:
    print(i, end='\r')
    s_val = str([tuple(x) for x in customer_research.loc[i:i + step].to_numpy()])[1:-1]
   
    #cur.execute(insert_s.replace('{s_val}',s_val))
    #conn.commit()
    ex_sql(insert_s.replace('{s_val}',s_val))
    i += step+1

In [ ]:
# using %s to appending string 
print("Hello Python developers! Welcome to, %s!" % str) 
